In [4]:
!pip install tiktoken

In [5]:
import torch
import tiktoken

In [6]:
with open("the-verdict.txt", "r", encoding="utf-8") as verdict:
    raw_text = verdict.read()

In [7]:
from torch.utils.data import Dataset, DataLoader
import torch

class LLMDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_len, stride) -> None:
     self.input_ids = []
     self.target_ids = []

     # Tokenize entire text
     # Changed from list [] to set {} to fix the TypeError
     token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

     # sliding window for making tensor x and y
     for i in range(0, len(token_ids) - max_len, stride):
      x = token_ids[i:i + max_len]
      y = token_ids[i + 1:i + max_len + 1]
      self.input_ids.append(torch.tensor(x))
      self.target_ids.append(torch.tensor(y))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [8]:
# Now well use the LLMDatasetV1 to load the inputs in batches via Pytorch DataLoader

# Step 1: Initialize the tokenizer
# Step 2: Create the dataset
# Step 3: drop_last = True drops the last batch if it's shorter
#         than the specified batch_size to prevent loss spikes
#         during training
# Step 4: The numer of CPU process to use for processing


def create_dataloader_v1(txt, batch_size=4, max_len=128,
                         stride=64, shuffle=True,
                         drop_last=True, num_workers=0):
  # Init the dataset
  tokenizer = tiktoken.get_encoding("gpt2")

  dataset = LLMDatasetV1(txt, tokenizer, max_len, stride)
  dataloader = DataLoader(dataset,
                          batch_size = batch_size,
                          shuffle = True,
                          drop_last = True
  )

  return dataloader

In [9]:
vocab_size = 50257
output_dims = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dims)

In [10]:
max_length = 4
dataloader = create_dataloader_v1(raw_text,
                                  batch_size=8,
                                  max_len=max_length,
                                  stride=max_length,
                                  shuffle=False,
                                  )
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [11]:
print("Token ID:\n", inputs)
print("\nInput Shape:\n", inputs.shape)

Token ID:
 tensor([[ 2077,   351,   257, 17275],
        [ 2952,    13,   198,   198],
        [49025,  5330, 15910,    13],
        [  691,  8208,    12, 14337],
        [10899,  2138,   257,  7026],
        [ 3499,  1466,    25,   484],
        [49655,  2651,    13,   366],
        [  355,  1752,   530,   550]])

Input Shape:
 torch.Size([8, 4])


In [12]:
token_embedding = token_embedding_layer(inputs)
print("Embedding Shape:\n", token_embedding.shape)

Embedding Shape:
 torch.Size([8, 4, 256])


In [13]:
context_length = max_length

pos_embedding_layer = torch.nn.Embedding(context_length, output_dims)
pos_embedding = pos_embedding_layer(torch.arange(max_length))
print("Positional Embedding Shape:\n", pos_embedding.shape)

Positional Embedding Shape:
 torch.Size([4, 256])


In [14]:
input_embedding = pos_embedding + token_embedding
print("Input Embedding Shape:\n", input_embedding.shape)

Input Embedding Shape:
 torch.Size([8, 4, 256])
